# StandUp4AI Evaluation - Fixed

**Key fix:** Skip F0 for short segments (< 0.5s). Use energy/ZCR/spectral instead.

**Expected:** ~800+ segments from 32 videos.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os

# Find standup4ai folder
base_paths = [
    '/content/drive/MyDrive/standup4ai',
    '/content/drive/Shareddrives/standup4ai',
]
BASE = None
for p in base_paths:
    if os.path.exists(p):
        BASE = p
        break

if BASE:
    AUDIO_DIR = os.path.join(BASE, 'audio')
    LABELS_DIR = os.path.join(BASE, 'labels')
    print(f'BASE: {BASE}')
    print(f'Audio: {os.path.exists(AUDIO_DIR)} ({len(os.listdir(AUDIO_DIR))} files)')
    print(f'Labels: {os.path.exists(LABELS_DIR)} ({len(os.listdir(LABELS_DIR))} files)')
else:
    print('ERROR: standup4ai folder not found!')
    print('Contents of MyDrive:')
    for item in sorted(os.listdir('/content/drive/MyDrive'))[:20]:
        print(f'  {item}')

In [ ]:
# 2. Find overlap
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a') or f.endswith('.mp3')]
label_files = [f for f in os.listdir(LABELS_DIR) if f.endswith('.csv')]

audio_vids = {f.replace('.m4a','').replace('.mp3','') for f in audio_files}
label_vids = {f.replace('.csv','') for f in label_files}
overlap = sorted(audio_vids & label_vids)

print(f'Audio: {len(audio_files)} files')
print(f'Labels: {len(label_files)} files')
print(f'Overlap: {len(overlap)} videos')
print(f'Sample: {overlap[:5]}')

In [ ]:
# 3. Install librosa
import subprocess
subprocess.run(['pip', 'install', '-q', 'librosa'], check=True, timeout=60)
import librosa
import numpy as np
import pandas as pd
print('librosa ready')

In [ ]:
# 4. Extract features - NO F0 for short segments!
def extract_features(audio_path, t0, t1, label):
    """Extract prosody features WITHOUT F0 (pyin fails on short segments)."""
    dur = t1 - t0
    
    # For very short segments, skip audio loading
    if dur < 0.1:
        return None
    
    try:
        # Load audio - limit to 10s max
        y, sr = librosa.load(audio_path, sr=22050, offset=t0, duration=min(dur, 10.0), mono=True)
    except:
        return None
    
    if len(y) < sr * 0.05:  # < 50ms
        return None
    
    feat = []
    
    # RMS energy (WORKS on all segments)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    feat.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.median(rms)])
    
    # ZCR (WORKS on all segments)
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
    feat.extend([np.mean(zcr), np.std(zcr), np.max(zcr)])
    
    # Spectral centroid
    cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
    feat.extend([np.mean(cent), np.std(cent)])
    
    # Spectral bandwidth
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
    feat.extend([np.mean(bw), np.std(bw)])
    
    # Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop)[0]
    feat.extend([np.mean(rolloff), np.std(rolloff)])
    
    # Spectral flatness
    flatness = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
    feat.extend([np.mean(flatness), np.std(flatness)])
    
    # Duration (normalized)
    feat.append(len(y) / sr)  # actual duration
    
    # FOR F0: Only attempt on segments >= 0.5s
    if dur >= 0.5:
        try:
            f0, voiced_flag, _ = librosa.pyin(y, fmin=80, fmax=500, sr=sr, hop_length=512)
            f0 = np.nan_to_num(f0, nan=0)
            voiced = np.nan_to_num(voiced_flag, nan=0)
            feat.extend([np.mean(f0), np.std(f0), np.mean(voiced)])
        except:
            feat.extend([0, 0, 0])  # F0 fails
    else:
        feat.extend([0, 0, 0])  # Too short for F0
    
    return np.array(feat, dtype=np.float32)

# Test on one segment
test_vid = overlap[0]
df = pd.read_csv(os.path.join(LABELS_DIR, f'{test_vid}.csv'))
row = df.iloc[0]
test_feat = extract_features(
    os.path.join(AUDIO_DIR, f'{test_vid}.m4a'),
    float(row['t0']), float(row['t1']), row['label']
)
print(f'Test feature dim: {len(test_feat)}')
print(f'Sample: {test_feat[:5]}')

In [ ]:
# 5. Full extraction
print(f'Extracting from {len(overlap)} videos...')

X_all, y_all, vids_all, durs_all = [], [], [], []
errors = 0

for i, vid in enumerate(overlap):
    audio_path = os.path.join(AUDIO_DIR, f'{vid}.m4a')
    label_path = os.path.join(LABELS_DIR, f'{vid}.csv')
    
    if not os.path.exists(audio_path) or not os.path.exists(label_path):
        continue
    
    try:
        df = pd.read_csv(label_path)
    except:
        continue
    
    for _, seg in df.iterrows():
        feat = extract_features(
            audio_path,
            float(seg['t0']), float(seg['t1']),
            seg['label']
        )
        if feat is None:
            errors += 1
            continue
        
        X_all.append(feat)
        y_all.append(1 if seg['label'].strip() == 'risa' else 0)
        vids_all.append(vid)
        durs_all.append(float(seg['t1']) - float(seg['t0']))
    
    if (i + 1) % 10 == 0:
        pos = sum(y_all)
        print(f'  {i+1}/{len(overlap)}: {len(X_all)} samples, {pos} pos ({100*pos/len(y_all):.1f}%)')

print(f'\n=== EXTRACTION DONE ===')
print(f'Total samples: {len(X_all)}')
print(f'Skipped (errors): {errors}')
if y_all:
    pos = sum(y_all)
    print(f'Positive: {pos} ({100*pos/len(y_all):.1f}%)')
print(f'Unique videos: {len(set(vids_all))}')
print(f'Feature dim: {X_all[0].shape if X_all else 0}')

In [ ]:
# 6. Training with video-level CV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import GroupKFold
import json

X = np.array(X_all)
y = np.array(y_all)
vids = np.array(vids_all)

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Positive rate: {100*y.mean():.1f}%')

# Video-level GroupKFold
n_splits = min(5, len(set(vids)))
gkf = GroupKFold(n_splits=n_splits)

models = {
    'LogReg': LogisticRegression(max_iter=2000, class_weight='balanced', C=0.1, solver='lbfgs'),
    'XGBoost': GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42),
}

results = {}
for name, model in models.items():
    f1s, precs, recs = [], [], []
    for tr, te in gkf.split(X, y, vids):
        if len(set(y[te])) < 2:
            continue
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        model.fit(Xtr, y[tr])
        pred = model.predict(Xte)
        f1s.append(f1_score(y[te], pred, zero_division=0))
        precs.append(precision_score(y[te], pred, zero_division=0))
        recs.append(recall_score(y[te], pred, zero_division=0))
    
    mean_f1 = np.mean(f1s) if f1s else 0
    std_f1 = np.std(f1s) if f1s else 0
    results[name] = {'f1': mean_f1, 'std': std_f1, 'prec': np.mean(precs), 'rec': np.mean(recs)}
    print(f'{name:12s} F1={mean_f1:.4f} ± {std_f1:.4f}  P={np.mean(precs):.4f} R={np.mean(recs):.4f}')

print(f'\n{"="*50}')
print(f'StandUp4AI baseline: F1=0.51')
print(f'{"="*50}')
for name, r in sorted(results.items(), key=lambda x: -x[1]['f1']):
    beat = '🏆 BEATS BASELINE' if r['f1'] > 0.51 else ''
    print(f'{name:12s} F1={r["f1"]:.4f} ± {r["std"]:.4f}  {beat}')

In [ ]:
# 7. Save results
result_data = {
    'n_samples': len(X_all),
    'n_videos': len(set(vids_all)),
    'positive_rate': float(y.mean()),
    'feature_dim': X.shape[1],
    'baseline_f1': 0.51,
    'results': {k: {'f1': v['f1'], 'std': v['std'], 'prec': v['prec'], 'rec': v['rec']} for k, v in results.items()}
}

result_path = os.path.join(BASE, 'standup4ai_results.json')
with open(result_path, 'w') as f:
    json.dump(result_data, f, indent=2)

print(f'\nSaved to: {result_path}')
print(json.dumps(result_data, indent=2))